In [86]:
import duckdb
import pandas as pd

In [87]:
df = pd.read_csv("data/processed/churn_clean.csv")

print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

df.head()

Rows : 7043
Columns : 50


,customer_id,count,country,state,city,zip_code,lat_long,latitude,longitude,gender,...,internet_category,service_count,engagement_score,customer_status,location,avg_revenue_per_month,coordinates,churn_reason_category,loyalty_level,customer_segment
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,DSL,3,48.19,Inactive,"Los Angeles, California",54.075000,"(33.964131, -118.272783)",Churned,New,At Risk
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Fiber,1,32.81,Inactive,"Los Angeles, California",75.825000,"(34.059281, -118.30742)",Churned,New,Standard
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Fiber,5,81.92,Inactive,"Los Angeles, California",102.562500,"(34.048013, -118.293953)",Churned,New,VIP At Risk
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Fiber,6,91.23,Inactive,"Los Angeles, California",108.787500,"(34.062125, -118.315709)",Churned,Regular,VIP At Risk
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Fiber,6,103.00,Inactive,"Los Angeles, California",102.781633,"(34.039224, -118.266293)",Churned,Loyal,VIP At Risk


In [88]:
con = duckdb.connect("database/customer_churn.duckdb")

In [89]:
con.register("customer_churn", df)

In [91]:
query = """
CREATE OR REPLACE VIEW churn_summary AS

SELECT

customer_id,
state,
city,
gender,
contract,
payment_method,
customer_segment,
loyalty_level,
monthly_charges,
total_charges,
cltv,
engagement_score,
churn_label

FROM customer_churn;
"""

con.execute(query)

print("View created successfully.")

View created successfully.


In [92]:
query = """
CREATE OR REPLACE VIEW revenue_summary AS

SELECT

state,

SUM(total_charges) AS revenue,

AVG(cltv) AS avg_cltv,

COUNT(*) AS customers

FROM customer_churn

GROUP BY state;
"""

con.execute(query)

print("Revenue view created.")

Revenue view created.


In [ ]:
query = """
SELECT *
FROM customer_churn
LIMIT 5;
"""

con.sql(query).df()

,customer_id,count,country,state,city,zip_code,lat_long,latitude,longitude,gender,...,internet_category,service_count,engagement_score,customer_status,location,avg_revenue_per_month,coordinates,churn_reason_category,loyalty_level,customer_segment
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,DSL,3,48.19,Inactive,"Los Angeles, California",54.075000,"(33.964131, -118.272783)",Churned,New,At Risk
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Fiber,1,32.81,Inactive,"Los Angeles, California",75.825000,"(34.059281, -118.30742)",Churned,New,Standard
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Fiber,5,81.92,Inactive,"Los Angeles, California",102.562500,"(34.048013, -118.293953)",Churned,New,VIP At Risk
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Fiber,6,91.23,Inactive,"Los Angeles, California",108.787500,"(34.062125, -118.315709)",Churned,Regular,VIP At Risk
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Fiber,6,103.00,Inactive,"Los Angeles, California",102.781633,"(34.039224, -118.266293)",Churned,Loyal,VIP At Risk


In [ ]:
query = """
SELECT COUNT(*) AS total_records
FROM customer_churn;
"""

con.sql(query).df()

,total_records
0,7043


In [ ]:
len(df.columns)

50

In [ ]:
query = """
SELECT
customer_id,
COUNT(*) AS duplicate_count
FROM customer_churn
GROUP BY customer_id
HAVING COUNT(*) > 1;
"""

con.sql(query).df()

,customer_id,duplicate_count


In [ ]:
query = """
SELECT COUNT(*) AS missing_cltv
FROM customer_churn
WHERE cltv IS NULL;
"""

con.sql(query).df()

,missing_cltv
0,0


In [ ]:
query = """
SELECT COUNT(*) AS missing_total_charges
FROM customer_churn
WHERE total_charges IS NULL;
"""

con.sql(query).df()

,missing_total_charges
0,11


In [ ]:
query = """
SELECT COUNT(*) AS negative_revenue
FROM customer_churn
WHERE total_charges < 0;
"""

con.sql(query).df()

,negative_revenue
0,0


In [ ]:
query = """
SELECT
churn_label,
COUNT(*) AS customers
FROM customer_churn
GROUP BY churn_label;
"""

con.sql(query).df()

,churn_label,customers
0,No,5174
1,Yes,1869


In [ ]:
query = """
SELECT
contract,
COUNT(*) AS customers
FROM customer_churn
GROUP BY contract
ORDER BY customers DESC;
"""

con.sql(query).df()

,contract,customers
0,Month-to-month,3875
1,Two year,1695
2,One year,1473


In [ ]:
query = """
SELECT
internet_service,
COUNT(*) AS customers
FROM customer_churn
GROUP BY internet_service
ORDER BY customers DESC;
"""

con.sql(query).df()

,internet_service,customers
0,Fiber optic,3096
1,DSL,2421
2,No,1526


In [ ]:
dictionary = pd.DataFrame({

"Column":df.columns,

"Data Type":[str(dtype) for dtype in df.dtypes]

})

dictionary

,Column,Data Type
0,customer_id,str
1,count,int64
2,country,str
3,state,str
4,city,str
5,zip_code,int64
6,lat_long,str
7,latitude,float64
8,longitude,float64
9,gender,str


In [ ]:
query = """
SELECT
COUNT(*) AS total_customers
FROM customer_churn;
"""

con.sql(query).df()

,total_customers
0,7043


In [ ]:
query = """
SELECT
COUNT(*) AS churned_customers
FROM customer_churn
WHERE churn_label='Yes';
"""

con.sql(query).df()

,churned_customers
0,1869


In [ ]:
query = """
SELECT

ROUND(

100.0*

SUM(

CASE

WHEN churn_label='Yes'

THEN 1

ELSE 0

END

)

/COUNT(*),2)

AS churn_rate

FROM customer_churn;
"""

con.sql(query).df()

,churn_rate
0,26.54


In [ ]:
query = """
SELECT

ROUND(

AVG(monthly_charges),2)

AS average_monthly_charge

FROM customer_churn;
"""

con.sql(query).df()

,average_monthly_charge
0,64.76


In [ ]:
query = """
SELECT

ROUND(

AVG(cltv),2)

AS average_cltv

FROM customer_churn;
"""

con.sql(query).df()

,average_cltv
0,4400.3


In [ ]:
query = """
SELECT

ROUND(

AVG(tenure_months),2)

AS average_tenure

FROM customer_churn;
"""

con.sql(query).df()

,average_tenure
0,32.37


In [ ]:
query = """
SELECT

gender,

COUNT(*) AS customers

FROM customer_churn

GROUP BY gender

ORDER BY customers DESC;
"""

con.sql(query).df()

,gender,customers
0,Male,3555
1,Female,3488


In [ ]:
query = """
SELECT

contract,

COUNT(*) AS customers

FROM customer_churn

GROUP BY contract

ORDER BY customers DESC;
"""

con.sql(query).df()

,contract,customers
0,Month-to-month,3875
1,Two year,1695
2,One year,1473


In [ ]:
query = """
SELECT

payment_method,

COUNT(*) AS customers

FROM customer_churn

GROUP BY payment_method

ORDER BY customers DESC;
"""

con.sql(query).df()

,payment_method,customers
0,Electronic check,2365
1,Mailed check,1612
2,Bank transfer (automatic),1544
3,Credit card (automatic),1522


In [ ]:
query = """
SELECT

customer_segment,

COUNT(*) AS customers

FROM customer_churn

GROUP BY customer_segment

ORDER BY customers DESC;
"""

con.sql(query).df()

,customer_segment,customers
0,VIP,2492
1,Standard,2280
2,At Risk,1241
3,VIP At Risk,1030


In [ ]:
query = """
SELECT

loyalty_level,

COUNT(*) AS customers

FROM customer_churn

GROUP BY loyalty_level

ORDER BY customers DESC;
"""

con.sql(query).df()

,loyalty_level,customers
0,New,2175
1,Regular,1856
2,Loyal,1594
3,Highly Loyal,1407
4,NaN,11


In [ ]:
query = """
SELECT

internet_service,

COUNT(*) AS customers

FROM customer_churn

GROUP BY internet_service

ORDER BY customers DESC;
"""

con.sql(query).df()

,internet_service,customers
0,Fiber optic,3096
1,DSL,2421
2,No,1526


In [ ]:
query = """
SELECT

state,

COUNT(*) customers

FROM customer_churn

GROUP BY state

ORDER BY customers DESC

LIMIT 10;
"""

con.sql(query).df()

,state,customers
0,California,7043


In [ ]:
query="""
SELECT

contract,

COUNT(*) customers,

SUM(CASE
WHEN churn_label='Yes'
THEN 1
ELSE 0
END) churned,

ROUND(

100.0*

SUM(
CASE
WHEN churn_label='Yes'
THEN 1
ELSE 0
END
)

/COUNT(*),2)

AS churn_rate

FROM customer_churn

GROUP BY contract

ORDER BY churn_rate DESC;
"""

con.sql(query).df()

,contract,customers,churned,churn_rate
0,Month-to-month,3875,1655.0,42.71
1,One year,1473,166.0,11.27
2,Two year,1695,48.0,2.83


In [ ]:
query="""
SELECT

payment_method,

COUNT(*) customers,

SUM(CASE
WHEN churn_label='Yes'
THEN 1
ELSE 0
END) churned,

ROUND(

100.0*

SUM(
CASE
WHEN churn_label='Yes'
THEN 1
ELSE 0
END
)

/COUNT(*),2)

AS churn_rate

FROM customer_churn

GROUP BY payment_method

ORDER BY churn_rate DESC;
"""

con.sql(query).df()

,payment_method,customers,churned,churn_rate
0,Electronic check,2365,1071.0,45.29
1,Mailed check,1612,308.0,19.11
2,Bank transfer (automatic),1544,258.0,16.71
3,Credit card (automatic),1522,232.0,15.24


In [ ]:
query="""
SELECT

customer_segment,

ROUND(

SUM(total_charges),2)

AS revenue

FROM customer_churn

GROUP BY customer_segment

ORDER BY revenue DESC;
"""

con.sql(query).df()

,customer_segment,revenue
0,VIP,7747366.30
1,Standard,4073357.95
2,VIP At Risk,2449899.25
3,At Risk,1785545.20


In [ ]:
query="""
SELECT

state,

ROUND(

SUM(total_charges),2)

AS revenue

FROM customer_churn

GROUP BY state

ORDER BY revenue DESC

LIMIT 10;
"""

con.sql(query).df()

,state,revenue
0,California,16056168.7


In [ ]:
query="""
SELECT

customer_id,

cltv,

customer_segment,

tenure_months

FROM customer_churn

ORDER BY cltv DESC

LIMIT 10;
"""

con.sql(query).df()

,customer_id,cltv,customer_segment,tenure_months
0,7622-FWGEW,6500,VIP,56
1,6024-RUGGH,6499,VIP,72
2,0383-CLDDA,6499,VIP,69
3,2683-JXWQQ,6495,VIP,61
4,8894-JVDCV,6494,VIP,62
5,4114-QMKVN,6494,VIP,56
6,4531-AUZNK,6492,VIP,51
7,1658-XUHBX,6492,VIP,59
8,2181-TIDSV,6492,VIP,68
9,0675-NCDYU,6491,VIP At Risk,72


In [ ]:
query="""
SELECT

customer_id,

cltv

FROM customer_churn

WHERE cltv >

(

SELECT AVG(cltv)

FROM customer_churn

)

ORDER BY cltv DESC;
"""

con.sql(query).df()

,customer_id,cltv
0,7622-FWGEW,6500
1,6024-RUGGH,6499
2,0383-CLDDA,6499
3,2683-JXWQQ,6495
4,8894-JVDCV,6494
...,...,...
3812,3255-GRXMG,4403
3813,8410-BGQXN,4402
3814,6510-UPNKS,4402
3815,0847-HGRML,4402


In [ ]:
query="""
SELECT

age_group,

COUNT(*) customers,

SUM(

CASE

WHEN churn_label='Yes'

THEN 1

ELSE 0

END

)

AS churned

FROM customer_churn

GROUP BY age_group

ORDER BY churned DESC;
"""

con.sql(query).df()

,age_group,customers,churned
0,Adult,5901,1393.0
1,Senior Citizen,1142,476.0


In [ ]:
query="""
SELECT

customer_segment,

ROUND(

AVG(cltv),2)

AS average_cltv

FROM customer_churn

GROUP BY customer_segment

ORDER BY average_cltv DESC;
"""

con.sql(query).df()

,customer_segment,average_cltv
0,VIP,5402.87
1,VIP At Risk,5357.71
2,Standard,3438.53
3,At Risk,3359.42


In [ ]:
query="""
SELECT

churn_reason_category,

COUNT(*) customers

FROM customer_churn

WHERE churn_label='Yes'

GROUP BY churn_reason_category

ORDER BY customers DESC;
"""

con.sql(query).df()

,churn_reason_category,customers
0,Churned,1869


In [ ]:
query="""
SELECT

state,

COUNT(*) customers

FROM customer_churn

GROUP BY state

HAVING COUNT(*)>100

ORDER BY customers DESC;
"""

con.sql(query).df()

,state,customers
0,California,7043


In [ ]:
query = """
WITH ranked_customers AS
(
SELECT

customer_id,
customer_segment,
cltv,

ROW_NUMBER()
OVER(
ORDER BY cltv DESC
) AS customer_rank

FROM customer_churn
)

SELECT *

FROM ranked_customers

WHERE customer_rank<=10;
"""

con.sql(query).df()

,customer_id,customer_segment,cltv,customer_rank
0,7622-FWGEW,VIP,6500,1
1,6024-RUGGH,VIP,6499,2
2,0383-CLDDA,VIP,6499,3
3,2683-JXWQQ,VIP,6495,4
4,8894-JVDCV,VIP,6494,5
5,4114-QMKVN,VIP,6494,6
6,1658-XUHBX,VIP,6492,7
7,4531-AUZNK,VIP,6492,8
8,2181-TIDSV,VIP,6492,9
9,0675-NCDYU,VIP At Risk,6491,10


In [ ]:
query = """
SELECT

state,

ROUND(
SUM(total_charges),2
) revenue,

RANK()
OVER(
ORDER BY SUM(total_charges) DESC
) revenue_rank

FROM customer_churn

GROUP BY state;
"""

con.sql(query).df()

,state,revenue,revenue_rank
0,California,16056168.7,1


In [ ]:
query="""
SELECT

customer_segment,

ROUND(
AVG(cltv),2
) average_cltv,

DENSE_RANK()
OVER(
ORDER BY AVG(cltv) DESC
) segment_rank

FROM customer_churn

GROUP BY customer_segment;
"""

con.sql(query).df()

,customer_segment,average_cltv,segment_rank
0,VIP,5402.87,1
1,VIP At Risk,5357.71,2
2,Standard,3438.53,3
3,At Risk,3359.42,4


In [ ]:
query="""
SELECT

customer_id,
cltv,

NTILE(4)
OVER(
ORDER BY cltv DESC
) cltv_quartile

FROM customer_churn;
"""

con.sql(query).df()

,customer_id,cltv,cltv_quartile
0,7622-FWGEW,6500,1
1,6024-RUGGH,6499,1
2,0383-CLDDA,6499,1
3,2683-JXWQQ,6495,1
4,8894-JVDCV,6494,1
...,...,...,...
7038,7928-VJYAB,2004,4
7039,4657-FWVFY,2004,4
7040,0871-URUWO,2003,4
7041,4925-LMHOK,2003,4


In [ ]:
query="""
WITH revenue AS
(

SELECT

state,

SUM(total_charges) revenue

FROM customer_churn

GROUP BY state

)

SELECT

state,
revenue,

SUM(revenue)
OVER(
ORDER BY revenue DESC
)

AS cumulative_revenue

FROM revenue;
"""

con.sql(query).df()

,state,revenue,cumulative_revenue
0,California,16056168.7,16056168.7


In [ ]:
query="""
SELECT

state,

ROUND(
SUM(total_charges),2
) revenue,

ROUND(

100*

SUM(total_charges)

/SUM(
SUM(total_charges)
)

OVER(),2)

AS revenue_percentage

FROM customer_churn

GROUP BY state

ORDER BY revenue_percentage DESC;
"""


con.sql(query).df()

,state,revenue,revenue_percentage
0,California,16056168.7,100.0


In [ ]:
query="""
SELECT

customer_id,

cltv,

LAG(cltv)
OVER(
ORDER BY cltv DESC
)

AS previous_cltv

FROM customer_churn;
"""

con.sql(query).df()

,customer_id,cltv,previous_cltv
0,7622-FWGEW,6500,<NA>
1,6024-RUGGH,6499,6500
2,0383-CLDDA,6499,6499
3,2683-JXWQQ,6495,6499
4,8894-JVDCV,6494,6495
...,...,...,...
7038,7928-VJYAB,2004,2004
7039,4657-FWVFY,2004,2004
7040,0871-URUWO,2003,2004
7041,4925-LMHOK,2003,2003


In [ ]:
query="""
SELECT

customer_id,

cltv,

LEAD(cltv)
OVER(
ORDER BY cltv DESC
)

AS next_cltv

FROM customer_churn;
"""

con.sql(query).df()

,customer_id,cltv,next_cltv
0,7622-FWGEW,6500,6499
1,6024-RUGGH,6499,6499
2,0383-CLDDA,6499,6495
3,2683-JXWQQ,6495,6494
4,8894-JVDCV,6494,6494
...,...,...,...
7038,7928-VJYAB,2004,2004
7039,4657-FWVFY,2004,2003
7040,0871-URUWO,2003,2003
7041,4925-LMHOK,2003,2003


In [ ]:
query="""
SELECT

customer_id,

cltv,

ROUND(

cltv-

AVG(cltv)
OVER(),2)

difference_from_average

FROM customer_churn

ORDER BY difference_from_average DESC;
"""

con.sql(query).df()

,customer_id,cltv,difference_from_average
0,7622-FWGEW,6500,2099.7
1,0383-CLDDA,6499,2098.7
2,6024-RUGGH,6499,2098.7
3,2683-JXWQQ,6495,2094.7
4,4114-QMKVN,6494,2093.7
...,...,...,...
7038,7928-VJYAB,2004,-2396.3
7039,0247-SLUJI,2004,-2396.3
7040,0871-URUWO,2003,-2397.3
7041,4925-LMHOK,2003,-2397.3


In [ ]:
query="""
WITH city_revenue AS
(

SELECT

state,
city,

SUM(total_charges) revenue,

ROW_NUMBER()

OVER(

PARTITION BY state

ORDER BY SUM(total_charges) DESC

) rn

FROM customer_churn

GROUP BY state,city

)

SELECT *

FROM city_revenue

WHERE rn=1;
"""

con.sql(query).df()

,state,city,revenue,rn
0,California,Los Angeles,647751.25,1


In [ ]:
query="""
SELECT

state,

ROUND(

100.0*

SUM(

CASE

WHEN churn_label='Yes'

THEN 1

ELSE 0

END

)

/COUNT(*),2)

AS churn_rate

FROM customer_churn

GROUP BY state

ORDER BY churn_rate DESC

LIMIT 10;
"""

con.sql(query).df()

,state,churn_rate
0,California,26.54


In [ ]:
query="""
SELECT

customer_id,

customer_segment,

cltv,

total_charges,

tenure_months,

engagement_score

FROM customer_churn

WHERE

high_value_customer='Yes'

AND

churn_label='Yes'

ORDER BY cltv DESC;
"""

con.sql(query).df()

,customer_id,customer_segment,cltv,total_charges,tenure_months,engagement_score
0,1043-YCUTE,VIP At Risk,6484,1327.15,56,97.24
1,1323-OOEPC,VIP At Risk,6481,5149.50,53,111.01
2,0112-QWPNC,VIP At Risk,6452,4059.35,49,119.12
3,5089-IFSDP,VIP At Risk,6424,6144.55,58,122.44
4,0406-BPDVR,VIP,6405,5373.10,54,110.65
...,...,...,...,...,...,...
785,9637-EIHEQ,VIP,4535,50.80,1,55.75
786,6317-YPKDH,VIP At Risk,4534,29.95,1,50.74
787,4910-AQFFX,VIP At Risk,4533,661.25,9,63.93
788,1976-CFOCS,VIP,4531,46.00,1,50.71


In [ ]:
query="""
SELECT

contract,

ROUND(

SUM(total_charges),2)

revenue_lost

FROM customer_churn

WHERE churn_label='Yes'

GROUP BY contract

ORDER BY revenue_lost DESC;
"""

con.sql(query).df()

,contract,revenue_lost
0,Month-to-month,1927182.25
1,One year,674991.20
2,Two year,260753.45


In [ ]:
query="""
SELECT

payment_method,

ROUND(

100.0*

SUM(

CASE

WHEN churn_label='Yes'

THEN 1

ELSE 0

END

)

/COUNT(*),2)

AS churn_rate

FROM customer_churn

GROUP BY payment_method

ORDER BY churn_rate DESC;
"""

con.sql(query).df()

,payment_method,churn_rate
0,Electronic check,45.29
1,Mailed check,19.11
2,Bank transfer (automatic),16.71
3,Credit card (automatic),15.24


In [ ]:
query="""
SELECT

state,

ROUND(
SUM(total_charges),2)
revenue,

ROUND(

100.0*

SUM(

CASE

WHEN churn_label='Yes'

THEN 1

ELSE 0

END

)

/COUNT(*),2)

AS churn_rate

FROM customer_churn

GROUP BY state

ORDER BY revenue DESC,
churn_rate DESC

LIMIT 10;
"""

con.sql(query).df()

,state,revenue,churn_rate
0,California,16056168.7,26.54


In [ ]:
query="""
SELECT

loyalty_level,

COUNT(*) customers,

ROUND(

SUM(total_charges),2)

revenue,

ROUND(

AVG(cltv),2)

average_cltv

FROM customer_churn

GROUP BY loyalty_level

ORDER BY revenue DESC;
"""

con.sql(query).df()

,loyalty_level,customers,revenue,average_cltv
0,Highly Loyal,1407,7289202.45,5214.58
1,Loyal,1594,5356180.85,4619.72
2,Regular,1856,2809133.50,4023.28
3,New,2175,601651.90,4038.16
4,NaN,11,NaN,3665.55


In [ ]:
query="""
SELECT

customer_segment,

ROUND(

AVG(cltv),2)

average_cltv,

ROUND(

SUM(total_charges),2)

total_revenue

FROM customer_churn

GROUP BY customer_segment

ORDER BY average_cltv DESC;
"""

con.sql(query).df()

,customer_segment,average_cltv,total_revenue
0,VIP,5402.87,7747366.30
1,VIP At Risk,5357.71,2449899.25
2,Standard,3438.53,4073357.95
3,At Risk,3359.42,1785545.20


In [ ]:
query="""
SELECT

COUNT(*) total_customers,

SUM(
CASE
WHEN churn_label='Yes'
THEN 1
ELSE 0
END
) churned_customers,

ROUND(

100.0*

SUM(
CASE
WHEN churn_label='Yes'
THEN 1
ELSE 0
END
)

/COUNT(*),2)

AS churn_rate,

ROUND(

AVG(cltv),2)

average_cltv,

ROUND(

AVG(monthly_charges),2)

average_monthly_charge,

ROUND(

SUM(total_charges),2)

total_revenue

FROM customer_churn;
"""

con.sql(query).df()

,total_customers,churned_customers,churn_rate,average_cltv,average_monthly_charge,total_revenue
0,7043,1869.0,26.54,4400.3,64.76,16056168.7


In [ ]:
query = """
SELECT

COUNT(*) AS total_customers,

SUM(
CASE
WHEN churn_label='Yes'
THEN 1
ELSE 0
END
) AS churned_customers,

ROUND(
100.0*
SUM(
CASE
WHEN churn_label='Yes'
THEN 1
ELSE 0
END
)
/COUNT(*),2
) AS churn_rate,

ROUND(AVG(monthly_charges),2) AS average_monthly_charge,

ROUND(AVG(cltv),2) AS average_cltv,

ROUND(SUM(total_charges),2) AS total_revenue

FROM customer_churn;
"""

dashboard_kpi = con.sql(query).df()

dashboard_kpi

,total_customers,churned_customers,churn_rate,average_monthly_charge,average_cltv,total_revenue
0,7043,1869.0,26.54,64.76,4400.3,16056168.7


In [ ]:
dashboard_kpi.to_csv("data/processed/dashboard_kpi.csv",index=False)

In [ ]:
query = """
SELECT

state,

ROUND(
SUM(total_charges),2
) revenue

FROM customer_churn

GROUP BY state

ORDER BY revenue DESC;
"""

dashboard_state = con.sql(query).df()

dashboard_state

,state,revenue
0,California,16056168.7


In [ ]:
dashboard_state.to_csv(
"data/processed/dashboard_state.csv",
index=False
)

In [ ]:

query = """
SELECT

customer_segment,

COUNT(*) customers

FROM customer_churn

GROUP BY customer_segment

ORDER BY customers DESC;
"""

dashboard_segment = con.sql(query).df()

dashboard_segment

,customer_segment,customers
0,VIP,2492
1,Standard,2280
2,At Risk,1241
3,VIP At Risk,1030


In [ ]:
dashboard_segment.to_csv(
"data/processed/dashboard_segment.csv",
index=False
)

In [ ]:
query = """
SELECT

contract,

COUNT(*) customers,

ROUND(
100.0*
SUM(
CASE
WHEN churn_label='Yes'
THEN 1
ELSE 0
END
)
/COUNT(*),2
) churn_rate

FROM customer_churn

GROUP BY contract;
"""

dashboard_contract = con.sql(query).df()

dashboard_contract

,contract,customers,churn_rate
0,One year,1473,11.27
1,Two year,1695,2.83
2,Month-to-month,3875,42.71


In [ ]:
dashboard_contract.to_csv(
"data/processed/dashboard_contract.csv",
index=False
)

In [ ]:
query = """
SELECT

loyalty_level,

ROUND(
SUM(total_charges),2
) revenue

FROM customer_churn

GROUP BY loyalty_level

ORDER BY revenue DESC;
"""

dashboard_loyalty = con.sql(query).df()

dashboard_loyalty

,loyalty_level,revenue
0,Highly Loyal,7289202.45
1,Loyal,5356180.85
2,Regular,2809133.50
3,New,601651.90
4,NaN,NaN


In [ ]:
dashboard_loyalty.to_csv(
"data/processed/dashboard_loyalty.csv",
index=False
)

In [ ]:
query = """
SELECT

age_group,

COUNT(*) customers

FROM customer_churn

GROUP BY age_group

ORDER BY age_group;
"""

dashboard_age = con.sql(query).df()

dashboard_age

,age_group,customers
0,Adult,5901
1,Senior Citizen,1142


In [ ]:
dashboard_age.to_csv(
"data/processed/dashboard_age.csv",
index=False
)

In [ ]:
query = """
SELECT

churn_reason_category,

COUNT(*) customers

FROM customer_churn

WHERE churn_label='Yes'

GROUP BY churn_reason_category

ORDER BY customers DESC;
"""

dashboard_reason = con.sql(query).df()

dashboard_reason

,churn_reason_category,customers
0,Churned,1869


In [ ]:
dashboard_reason.to_csv(
"data/processed/dashboard_reason.csv",
index=False
)

In [ ]:
query = """
SELECT

city,

ROUND(
SUM(total_charges),2
) revenue

FROM customer_churn

GROUP BY city

ORDER BY revenue DESC

LIMIT 10;
"""

dashboard_city = con.sql(query).df()

dashboard_city

,city,revenue
0,Los Angeles,647751.25
1,San Diego,354896.60
2,Sacramento,256295.05
3,San Jose,243735.55
4,San Francisco,221624.65
5,Fresno,154890.10
6,Long Beach,141928.85
7,Oakland,114974.75
8,Whittier,96832.30
9,Bakersfield,93714.40


In [ ]:
dashboard_city.to_csv(
"data/processed/dashboard_city.csv",
index=False
)

In [93]:
query = """

SELECT

customer_id,

state,

city,

contract,

customer_segment,

loyalty_level,

monthly_charges,

total_charges,

cltv,

engagement_score,

churn_label

FROM customer_churn;

"""

dashboard_master = con.sql(query).df()

dashboard_master

,customer_id,state,city,contract,customer_segment,loyalty_level,monthly_charges,total_charges,cltv,engagement_score,churn_label
0,3668-QPYBK,California,Los Angeles,Month-to-month,At Risk,New,53.85,108.15,3239,48.19,Yes
1,9237-HQITU,California,Los Angeles,Month-to-month,Standard,New,70.70,151.65,2701,32.81,Yes
2,9305-CDSKC,California,Los Angeles,Month-to-month,VIP At Risk,New,99.65,820.50,5372,81.92,Yes
3,7892-POOKP,California,Los Angeles,Month-to-month,VIP At Risk,Regular,104.80,3046.05,5003,91.23,Yes
4,0280-XJGEX,California,Los Angeles,Month-to-month,VIP At Risk,Loyal,103.70,5036.30,5340,103.00,Yes
...,...,...,...,...,...,...,...,...,...,...,...
7038,2569-WGERO,California,Landers,Two year,VIP,Highly Loyal,21.15,1419.40,5306,86.86,No
7039,6840-RESVB,California,Adelanto,One year,Standard,Regular,84.80,1990.50,2140,66.00,No
7040,2234-XADUH,California,Amboy,One year,VIP,Highly Loyal,103.20,7362.90,5560,114.40,No
7041,4801-JZAZL,California,Angelus Oaks,Month-to-month,Standard,New,29.60,346.45,2793,37.33,No
